In [64]:
import requests
import json
from dotenv import load_dotenv
import os
import urllib.parse
from openai import OpenAI

In [65]:
# Load .env from the current folder
load_dotenv()

True

In [66]:
CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
REDIRECT_URI = os.getenv("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:8000/callback")
SCOPES = "playlist-modify-private playlist-modify-public user-read-private"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [67]:
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_PROMPT = """
You are a music curator whose task is to suggest songs and artists based only on the user's full mood description. 

Rules:
1. Prioritize the user's full description.
2. Ignore popularity or mainstream success — small, indie, or lesser-known artists are extremely preferred.
3. Provide a mix of different artists that match the description. 
4. A total of 25 tracks. 
5. Return a Python dictionary (not JSON) with this structure:
   {"tracks": [{"artist": "Artist Name", "track": "Track Name"}, ...]}
"""

def get_tracks_from_gpt(user_prompt: str):
    """Ask GPT for tracks and return them directly as a Python dictionary."""
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": MODEL_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_tokens=600
    )
    
    # Use the model’s text directly — GPT will format it as valid Python
    text = response.choices[0].message.content.strip()

    # Evaluate safely to a Python dict
    try:
        # Use `ast.literal_eval` for safety instead of eval()
        import ast
        data = ast.literal_eval(text)
        return data
    except Exception as e:
        print(" Could not parse GPT response as Python dict.")
        print(text)
        raise e

In [68]:
user_mood = "jämtland mountain electronic female indie"
results_dict = get_tracks_from_gpt(user_mood)

# Inspect the results
print(type(results_dict))  # should say <class 'dict'>
print(len(results_dict["tracks"]), "tracks found")
print(results_dict["tracks"][:3])  # show first few

<class 'dict'>
25 tracks found
[{'artist': 'Hanna Järver', 'track': 'Anekdoter'}, {'artist': 'Klara & Jag', 'track': "Fake It 'Til You Make It"}, {'artist': 'Mira Aasma', 'track': 'Ghost'}]


In [69]:
# CELL 3 — Authorization Code Flow (manual step: open URL, copy code)

# Step A: build auth url
auth_url = "https://accounts.spotify.com/authorize?" + urllib.parse.urlencode({
    "client_id": CLIENT_ID,
    "response_type": "code",
    "redirect_uri": REDIRECT_URI,
    "scope": SCOPES,
    "show_dialog": "true"
})
print("1) Open this URL in your browser and authorize the app:\n")
print(auth_url)
print("\nAfter login Spotify will redirect to your REDIRECT_URI with a ?code=... parameter.")
code = input("2) Paste the 'code' value here: ").strip()

# Step B: Exchange code for tokens
token_url = "https://accounts.spotify.com/api/token"
resp = requests.post(token_url, data={
    "grant_type": "authorization_code",
    "code": code,
    "redirect_uri": REDIRECT_URI,
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET
})
resp.raise_for_status()
tokens = resp.json()
SPOTIFY_ACCESS_TOKEN = tokens["access_token"]
SPOTIFY_REFRESH_TOKEN = tokens.get("refresh_token")
print("Access token obtained (masked):", SPOTIFY_ACCESS_TOKEN[:6] + "...")  # don't print full token

# Step C: Verify token works with /v1/me
headers = {"Authorization": f"Bearer {SPOTIFY_ACCESS_TOKEN}"}
r = requests.get("https://api.spotify.com/v1/me", headers=headers)
r.raise_for_status()
me = r.json()
print("Authenticated as Spotify display name:", me.get("display_name"))
print("Spotify user id:", me.get("id"))

# Persist tokens in memory for subsequent cells
_spotify_auth = {
    "access_token": SPOTIFY_ACCESS_TOKEN,
    "refresh_token": SPOTIFY_REFRESH_TOKEN,
    "expires_in": tokens.get("expires_in")
}

1) Open this URL in your browser and authorize the app:

https://accounts.spotify.com/authorize?client_id=c00855f909a5419d802cb163c6fbfdf8&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8000%2Fcallback&scope=playlist-modify-private+playlist-modify-public+user-read-private&show_dialog=true

After login Spotify will redirect to your REDIRECT_URI with a ?code=... parameter.


2) Paste the 'code' value here:  AQBQfqS-GLKSTRyqQwSu5i9BSmc5Z2ur2YG652N3TMbsO7vAIv7W9wTA8BPqEkyqvI3BV-da7YTRSaus7T_mijFti3pZMqjrCmDoROXqmfJMsUkuWO3hLArcpCoWesyyTptkE4b-FDLMz7hObrBF3GUXGSHCORIolvycoSp182u4Fc0Mqlpqfv9fi2FrZ1LldhlkYd2gWSd-VHHy4i0Jbgs5AZch0CDZU-iK_CyI8DxySePKs6UQEpmp4PuesE0TIF-XUOJm


Access token obtained (masked): BQCOyT...
Authenticated as Spotify display name: Viktoria Sundelin
Spotify user id: viktoriasundelin


In [70]:
def search_spotify_track(track_name, artist_name, access_token):
    """
    Search Spotify for a track by name and artist.
    Returns a list of dicts with keys: name, artist, uri
    """
    import requests

    q = f"track:{track_name} artist:{artist_name}"
    url = "https://api.spotify.com/v1/search"
    headers = {"Authorization": f"Bearer {access_token}"}
    params = {"q": q, "type": "track", "limit": 3}

    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()

    items = r.json().get("tracks", {}).get("items", [])
    results = []

    for item in items:
        results.append({
            "name": item["name"],
            "artist": item["artists"][0]["name"],
            "uri": item["uri"],
            "id": item["id"]
        })

    return results

In [71]:
track_uris = []
artist_id = []

for t in results_dict["tracks"]:
    track_name = t["track"]
    artist_name = t["artist"]

    matches = search_spotify_track(track_name, artist_name, SPOTIFY_ACCESS_TOKEN)

    if matches:
        track_uris.append(matches[0]["uri"])  # first match's URI
        artist_id.append(matches[0]["id"])  # first match's artist_id
        print(f"Added: {matches[0]['artist']} - {matches[0]['name']}")
    else:
        print(f" Not found: {artist_name} - {track_name}")

print(f"\n Total tracks found: {len(track_uris)}")


Added: Hanna Järver - Anekdoter
Added: Klara & Jag - Fake It 'Til You Make It
 Not found: Mira Aasma - Ghost
Added: Alice Boman - Dreams
Added: Elsa & Emilie - Au Volant
Added: Ji Nilsson - Perfume
Added: Elliphant - Down On Life
Added: Lykke Li - I Follow Rivers
Added: Tove Styrke - Say My Name
Added: Seinabo Sey - Hard Time
Added: LÉON - Tired of Talking
Added: Molly Sandén - Freak
Added: Zara Larsson - Lush Life
Added: Miriam Bryant - Black Car
 Not found: Ebba Tove Elsa Nilsson - Stay High
Added: Amanda Mair - Wednesday
Added: Frida Sundemo - Gold
Added: Linnea Henriksson - Halmstad
Added: Veronica Maggio - Jag kommer
Added: Anna Ternheim - The Longer The Waiting (The Sweeter The Kiss)
Added: Miss Li - Beautiful
Added: Laleh - Some Die Young
Added: Marlene - Next To Me
Added: Ida Redig - Shout
 Not found: Ellen Sundberg - The Storm

 Total tracks found: 22


In [72]:
import requests

def create_spotify_playlist(user_id, name, description, access_token):
    """
    Create a private playlist on Spotify.
    Returns the playlist ID.
    """
    url = f"https://api.spotify.com/v1/users/{user_id}/playlists"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    body = {
        "name": name,
        "description": description,
        "public": False
    }
    r = requests.post(url, headers=headers, json=body)
    r.raise_for_status()
    return r.json()["id"]

In [73]:
SPOTIFY_USER_ID = "viktoriasundelin"

In [74]:
playlist_name = "Noridc Noir"
playlist_description = "Generated from GPT track suggestions"
playlist_id = create_spotify_playlist(SPOTIFY_USER_ID, playlist_name, playlist_description, SPOTIFY_ACCESS_TOKEN)
print("Playlist created, ID:", playlist_id)


Playlist created, ID: 7tmHKzzCbn2dWGUKTdPAtu


In [75]:
def add_tracks_to_playlist(playlist_id, track_uris, access_token):
    """
    Add a list of track URIs to a Spotify playlist.
    """
    url = f"https://api.spotify.com/v1/playlists/{playlist_id}/tracks"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    # Spotify API accepts up to 100 tracks per request
    chunk_size = 100
    for i in range(0, len(track_uris), chunk_size):
        chunk = track_uris[i:i+chunk_size]
        r = requests.post(url, headers=headers, json={"uris": chunk})
        r.raise_for_status()


In [76]:
add_tracks_to_playlist(playlist_id, track_uris, SPOTIFY_ACCESS_TOKEN)

print(f" {len(track_uris)} tracks added to playlist '{playlist_name}'")

 22 tracks added to playlist 'Noridc Noir'


In [77]:
# Compute Drift Index

In [87]:
from collections import Counter
import requests
import math

In [88]:
# --- Helper: fetch artist info ---
def get_spotify_artist(artist_id, access_token):
    url = f"https://api.spotify.com/v1/artists/{artist_id}"
    headers = {"Authorization": f"Bearer {access_token}"}
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    return r.json()

In [89]:
# --- Helper: fetch track info ---
def get_spotify_track(track_id, access_token):
    url = f"https://api.spotify.com/v1/tracks/{track_id}"
    headers = {"Authorization": f"Bearer {access_token}"}
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    return r.json()

In [90]:
# --- Indie Fraction (by popularity) ---
def indie_fraction(track_uris, access_token, threshold=50):
    """
    track_uris: list of track Spotify URIs
    threshold: popularity below which track is considered 'indie'
    Returns fraction of indie tracks
    """
    if not track_uris:
        return 0

    count_indie = 0
    total = 0

    for uri in track_uris:
        track_id = uri.split(":")[-1]  # extract ID from URI
        track_data = get_spotify_track(track_id, access_token)
        popularity = track_data.get("popularity", 0)
        total += 1
        if popularity < threshold:
            count_indie += 1

    return count_indie / total if total > 0 else 0.0

In [92]:
# --- Example Usage ---
diversity_score, genre_counts = genre_diversity(artist_ids, SPOTIFY_ACCESS_TOKEN)
indie_pct = indie_fraction(track_uris, SPOTIFY_ACCESS_TOKEN, threshold=50)

print(f"Idependant tracks fraction: {indie_pct*100:.1f}% (popularity < 50)")

 Idependant tracks fraction: 81.8% (popularity < 50)
